# Targets

This notebook is strictly for target construction and target plots. It rebuilds the feature set only to get the reference price and short implied-vol proxy needed for forward targets.

Default bucket size is 5 minutes.

In [ ]:
from __future__ import annotations

import os
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns

if "notebooks" not in sys.path:
    sys.path.append("notebooks")

from advanced_features import (
    build_feature_set,
    build_targets,
    discover_datasets,
    latest_feature_date,
)

ROOT = Path(os.environ.get("MODL_WS_NORMALIZED_DIR", "/mnt/burner-archive/ws_normalized")).expanduser()
DATE = os.environ.get("MODL_VIEW_DATE") or latest_feature_date(ROOT)
DATE_TAG = datetime.strptime(DATE, "%Y-%m-%d").strftime("%y-%m-%d")
FEATURE_ROOT = Path(os.environ.get("MODL_WS_FEATURE_DIR", "/mnt/burner-archive/ws_features")).expanduser()
BAR_MINUTES = int(os.environ.get("MODL_BAR_MINUTES", "5"))
HORIZONS = (5, 15, 30)
SAVE_OUTPUTS = False

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 240)
pd.set_option("display.max_colwidth", 180)
pl.Config.set_tbl_cols(240)
pl.Config.set_tbl_rows(24)

DATASETS = discover_datasets(ROOT, DATE_TAG)
if not DATASETS:
    raise FileNotFoundError(f"No normalized Parquet files found under {ROOT} for {DATE}")

ROOT, DATE, DATE_TAG, BAR_MINUTES, len(DATASETS)


## Build Feature Set And Targets

In [ ]:
feature_set = build_feature_set(DATASETS, horizons=HORIZONS, bar_minutes=BAR_MINUTES)
feature_matrix = feature_set.feature_matrix
base_feature_matrix = feature_set.base_feature_matrix
trade_features = feature_set.trade_features
book_features = feature_set.book_features
deribit_option_features = feature_set.deribit_option_features
term_structure_features = feature_set.term_structure
option_smile_features = feature_set.option_smile
futures_basis_features = feature_set.futures_basis
funding_features = feature_set.funding_features
rv_features = feature_set.rv_features
reference_price = feature_set.reference_price

df = feature_matrix
df_stats = df.describe().T

component_shapes = pd.DataFrame(
    [
        ("feature_matrix", *feature_matrix.shape),
        ("base_feature_matrix", *base_feature_matrix.shape),
        ("trade_features", trade_features.height, trade_features.width),
        ("book_features", book_features.height, book_features.width),
        ("deribit_option_features", *deribit_option_features.shape),
        ("term_structure_features", *term_structure_features.shape),
        ("option_smile_features", *option_smile_features.shape),
        ("futures_basis_features", *futures_basis_features.shape),
        ("funding_features", *funding_features.shape),
        ("rv_features", *rv_features.shape),
    ],
    columns=["component", "rows", "columns"],
)
component_shapes

In [ ]:
targets = build_targets(reference_price, term_structure_features, HORIZONS, bar_minutes=BAR_MINUTES)
target_stats = targets.describe().T
target_stats

## Target Table

In [ ]:
targets.tail(30)

## Forward Return Targets

In [ ]:
return_cols = [column for column in targets.columns if column.startswith("target_future_return_")]
fig, axes = plt.subplots(2, 1, figsize=(15, 9))
targets[return_cols].plot(ax=axes[0], marker="o")
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("Forward Log Return Targets")
targets[return_cols].hist(ax=axes[1], bins=20)
axes[1].set_title("Forward Return Distributions")
plt.tight_layout()

## Future Realized Volatility Targets

In [ ]:
rv_target_cols = [column for column in targets.columns if column.startswith("target_future_rv_")]
fig, axes = plt.subplots(2, 1, figsize=(15, 9))
targets[rv_target_cols].plot(ax=axes[0], marker="o")
axes[0].set_title("Forward Realized Volatility Targets")
targets[rv_target_cols].hist(ax=axes[1], bins=20)
axes[1].set_title("Forward RV Distributions")
plt.tight_layout()

## VRP Targets

In [ ]:
vrp_cols = [column for column in targets.columns if column.startswith("target_vrp_")]
fig, axes = plt.subplots(2, 1, figsize=(15, 9))
targets[vrp_cols].plot(ax=axes[0], marker="o")
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("VRP Targets: Short Implied Variance Minus Future Realized Variance")
targets[vrp_cols].hist(ax=axes[1], bins=20)
axes[1].set_title("VRP Target Distributions")
plt.tight_layout()

## FIR Execution Return Targets
These targets estimate the log PnL from entering after the current bar and exiting through a smoothed future execution schedule.

In [ ]:
fir_cols = [column for column in targets.columns if column.startswith("target_fir_return_")]
fir_cols

In [ ]:
if fir_cols:
    fig, axes = plt.subplots(2, 1, figsize=(15, 9))
    targets[fir_cols].plot(ax=axes[0], marker="o")
    axes[0].axhline(0, color="black", linewidth=1)
    axes[0].set_title("FIR Execution-Weighted Forward Return Targets")
    targets[fir_cols].hist(ax=axes[1], bins=20)
    axes[1].set_title("FIR Target Distributions")
    plt.tight_layout()
else:
    print("No FIR targets available")

## Target Availability

In [ ]:
availability = pd.DataFrame({"non_null": targets.notna().sum(), "null": targets.isna().sum(), "coverage": targets.notna().mean()}).sort_values("coverage", ascending=False)
availability

## Optional Save

In [ ]:
if SAVE_OUTPUTS:
    out_dir = FEATURE_ROOT / DATE / f"bar_{BAR_MINUTES}m"
    out_dir.mkdir(parents=True, exist_ok=True)
    targets.to_parquet(out_dir / "targets.parquet")
    print(f"wrote targets to {out_dir}")
else:
    print("SAVE_OUTPUTS is false; nothing written")